# Task 3: Real-Time Echocardiogram Video Analysis
Video: `data/echo.mp4` (0X100009310A3BD7FC.mp4 from Stanford EchoNet-Dynamic). Press **q** to stop.

In [1]:
import cv2
import numpy as np

## Pipeline (for every frame)
1. Histogram Equalization (fix murky contrast)
2. JET heatmap (show intensities as colors)
3. Color balance (gray world)
4. Log transform (reveal dark heart chambers)
5. Power-law with gamma = 1.5 (> 1, pushes down the bright white backscatter noise)

In [2]:
# Video Capture Setup (file instead of webcam)
video = cv2.VideoCapture("data/echo.mp4")

c = 255 / np.log(1 + 255)   # log constant
gamma = 1.5                 # gamma > 1 darkens the very bright noise
frame_count = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    frame_count = frame_count + 1

    # EchoNet videos are only 112 x 112, make them bigger to see
    frame = cv2.resize(frame, (336, 336))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 1. Histogram Equalization
    equalized = cv2.equalizeHist(gray)

    # 2. Colored heatmap
    heatmap = cv2.applyColorMap(equalized, cv2.COLORMAP_JET)

    # 3. Color balance (gray world)
    b, g, r = cv2.split(heatmap.astype(np.float32))
    avg = (b.mean() + g.mean() + r.mean()) / 3
    b = b * (avg / b.mean())
    g = g * (avg / g.mean())
    r = r * (avg / r.mean())
    balanced = np.clip(cv2.merge([b, g, r]), 0, 255).astype(np.uint8)

    # 4. Log transform
    log_frame = np.uint8(c * np.log(1 + balanced.astype(np.float32)))

    # 5. Power-law transform
    enhanced = np.uint8(255 * (log_frame / 255) ** gamma)

    # Monitoring array: raw | enhanced side by side
    combined = np.hstack([frame, enhanced])

    cv2.putText(combined, "Raw", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(combined, "Enhanced", (346, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.imshow("Echocardiogram: Raw vs Enhanced", combined)

    # Save one screenshot for the output folder
    if frame_count == 50:
        cv2.imwrite("output/side_by_side.png", combined)

    if cv2.waitKey(20) == ord("q"):
        break

video.release()
cv2.destroyAllWindows()

print("Frames processed:", frame_count)

Frames processed: 174
